<a href="https://colab.research.google.com/github/tarunrkk/RetouchAI/blob/main/RetouchAI_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch diffusers transformers accelerate pillow numpy

In [ ]:
# Import all the libraries
import torch
import numpy as np
from PIL import Image, ImageOps
from transformers import CLIPSegProcessor, CLIPSegForImageSegmentation
from diffusers import FluxInpaintPipeline
import os

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
# ---------------------------------------------------------
# 1. Initialize Device & Models
# ---------------------------------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading CLIPSeg for text-based object identification...")
processor = CLIPSegProcessor.from_pretrained("CIDAS/clipseg-rd64-refined")
segmentation_model = CLIPSegForImageSegmentation.from_pretrained("CIDAS/clipseg-rd64-refined").to(device)

print("Loading FLUX.1 [schnell] for stable, hyper-fast generative inpainting...")
# Swapped to FLUX.1-schnell which has native, stable inpainting support
inpaint_pipe = FluxInpaintPipeline.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    torch_dtype=torch.bfloat16
).to(device)

# Enable CPU offload to ensure it runs smoothly on Colab's 16GB GPUs
inpaint_pipe.enable_model_cpu_offload()

In [ ]:
# ---------------------------------------------------------
# 2. Define the Core Pipeline Function
# ---------------------------------------------------------
def generate_schnell_product_shot(image_path, object_prompt, background_prompt, seed=42):
    print(f"Loading image from: {image_path}")
    original_image = Image.open(image_path).convert("RGB")

    # Standardizing to 1024x1024
    original_image = original_image.resize((1024, 1024))

    # --- STEP A: Identify and Isolate the Object ---
    print(f"Identifying the '{object_prompt}'...")
    inputs = processor(
        text=[object_prompt],
        images=[original_image],
        padding="max_length",
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = segmentation_model(**inputs)

    tensor_mask = torch.sigmoid(outputs.logits)
    mask_np = tensor_mask.cpu().numpy()

    # Thresholding for a hard mask
    threshold = 0.4
    binary_mask = (mask_np > threshold).astype(np.uint8) * 255
    object_mask = Image.fromarray(binary_mask).convert("L")
    object_mask = object_mask.resize(original_image.size)

    # Invert the mask for the background replacement
    background_mask = ImageOps.invert(object_mask)

    # --- STEP B: Generate New Background with FLUX.1 Schnell ---
    print("Generating new environment in 4 steps with FLUX.1 Schnell...")
    generator = torch.Generator(device=device).manual_seed(seed)

    result = inpaint_pipe(
        prompt=background_prompt,
        image=original_image,
        mask_image=background_mask,
        num_inference_steps=4,   # Distilled model: exactly 4 steps required
        guidance_scale=0.0,      # Distilled model: guidance must be 0.0
        strength=0.99,           # Total background replacement
        generator=generator
    ).images[0]

    return result, background_mask# ---------------------------------------------------------
# 2. Define the Core Pipeline Function
# ---------------------------------------------------------
def generate_schnell_product_shot(image_path, object_prompt, background_prompt, seed=42):
    print(f"Loading image from: {image_path}")
    original_image = Image.open(image_path).convert("RGB")

    # Standardizing to 1024x1024
    original_image = original_image.resize((1024, 1024))

    # --- STEP A: Identify and Isolate the Object ---
    print(f"Identifying the '{object_prompt}'...")
    inputs = processor(
        text=[object_prompt],
        images=[original_image],
        padding="max_length",
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = segmentation_model(**inputs)

    tensor_mask = torch.sigmoid(outputs.logits)
    mask_np = tensor_mask.cpu().numpy()

    # Thresholding for a hard mask
    threshold = 0.4
    binary_mask = (mask_np > threshold).astype(np.uint8) * 255
    object_mask = Image.fromarray(binary_mask).convert("L")
    object_mask = object_mask.resize(original_image.size)

    # Invert the mask for the background replacement
    background_mask = ImageOps.invert(object_mask)

    # --- STEP B: Generate New Background with FLUX.1 Schnell ---
    print("Generating new environment in 4 steps with FLUX.1 Schnell...")
    generator = torch.Generator(device=device).manual_seed(seed)

    result = inpaint_pipe(
        prompt=background_prompt,
        image=original_image,
        mask_image=background_mask,
        num_inference_steps=4,   # Distilled model: exactly 4 steps required
        guidance_scale=0.0,      # Distilled model: guidance must be 0.0
        strength=0.99,           # Total background replacement
        generator=generator
    ).images[0]

    return result, background_mask

In [ ]:
# ---------------------------------------------------------
# 3. Execution Block
# ---------------------------------------------------------
if __name__ == "__main__":
    image_name = "CoffeeMug1"
    folder_raw = "ImagesRaw"
    image_type = "CoffeeMugs"
    image_path = f"/content/Images/{folder_raw}/{image_type}/{image_name}.jpg"

    folder_mask = "ImagesMasks"
    folder_output = "ImagesOutput"
    os.makedirs(f"/content/Images/{folder_mask}/{image_type}", exist_ok=True)
    os.makedirs(f"/content/Images/{folder_output}/{image_type}", exist_ok=True)

    prompt_path = f"/content/Prompts/{image_type}/bg_prompt1.txt"

    obj_prompt = "a coffee mug"
    bg_prompt = open(prompt_path, "r").read()

    try:
        final_image, generated_mask = generate_schnell_product_shot(
            image_path=image_path,
            object_prompt=obj_prompt,
            background_prompt=bg_prompt,
            seed=1024
        )

        final_image.save(f"/content/Images/{folder_output}/{image_type}/{image_name}.jpg")
        generated_mask.save(f"/content/Images/{folder_mask}/{image_type}/{image_name}.jpg")

        print(f"Success! Final image saved as 'schnell_studio_mug_output.png'.")

    except FileNotFoundError:
        print(f"Error: Could not find '{image_path}'. Please upload it to your Colab session.")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

An unexpected error occurred: name 'generate_klein_product_shot' is not defined
